# R03 Lever Ablations - H14 seeding, H15 decomposition, H16 head+tail

**Author**: Knowledge Graph Foundry autonomous build

Each lever toggled independently on the densified R01 graph with propositions on and the strong reader, abstention off (H17 already adjudicated). H15 measured on the 8 comparison probes only (its target population); H14 and H16 on all 24 answerable probes.

In [1]:
# imports
import datetime
import json
import re
from pathlib import Path

import yaml

from knowledge_graph_foundry import Foundry, load_settings

2026-07-06 14:23:02.546 | INFO     | knowledge_graph_foundry.config:<module>:40 - PROJ_ROOT path is: /home/lab/workspace/learning/projects/knowledge-graph-foundry


In [2]:
# probes + scoring (same rules as probe_eval.ipynb)
probes = yaml.safe_load(Path('../tests/probes/cpap-probe-set.yml').read_text())
answerable = [p for p in probes if p['category'] != 'unanswerable']
comparisons = [p for p in probes if p['category'] == 'comparison']

def _norm(s):
    return re.sub(r'\s+', ' ', s.casefold())

def evidence_recall(gold, context):
    ctx = _norm(context)
    return sum(1 for g in gold if _norm(g) in ctx) / len(gold) if gold else None

def value_tokens(gold_answer):
    return re.findall(r'[\w.\-/]*\d[\w.\-/]*', gold_answer)

def answer_correct(probe, answer):
    ans = _norm(answer)
    tokens = value_tokens(probe['gold_answer'])
    if tokens:
        hit = sum(1 for t in tokens if _norm(t) in ans)
        return hit >= max(1, len(tokens) // 2 + (len(tokens) % 2))
    return _norm(probe['gold_answer']) in ans

def base_settings():
    s = load_settings(Path('../config.yml') if Path('../config.yml').exists() else None)
    s.graphrag.propositions_enabled = True
    s.graphrag.abstention_enabled = False
    return s

def run(settings, probe_list, label, with_recall=True):
    rows = []
    with Foundry(settings) as f:
        for p in probe_list:
            rec = None
            if with_recall:
                lines, _, _ = f._retrieve_local(p['question'])
                rec = evidence_recall(p['gold_evidence'], '\n'.join(lines))
            ans = f.query(p['question'])['answer']
            rows.append({'id': p['id'], 'recall': rec,
                         'correct': answer_correct(p, ans), 'answer': ans})
            print(p['id'], rows[-1]['recall'], rows[-1]['correct'])
    recs = [r['recall'] for r in rows if r['recall'] is not None]
    return {'label': label, 'rows': rows,
            'recall': sum(recs) / len(recs) if recs else None,
            'accuracy': sum(r['correct'] for r in rows) / len(rows)}

In [3]:
# H14: proposition seeding off vs on (context identical, seeds differ)
s = base_settings(); s.graphrag.proposition_seeding = False
h14_off = run(s, answerable, 'H14_seeding_off')
s = base_settings(); s.graphrag.proposition_seeding = True
h14_on = run(s, answerable, 'H14_seeding_on')
print('H14:', h14_off['recall'], '->', h14_on['recall'],
      '| acc', h14_off['accuracy'], '->', h14_on['accuracy'])

Received notification from DBMS server: <GqlStatusObject gql_status='01N03', status_description='warn: procedure field deprecated. The field `schema` of procedure gds.graph.drop() is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL gds.graph.drop($name, false)'


Received notification from DBMS server: <GqlStatusObject gql_status='01N03', status_description='warn: procedure field deprecated. The field `schema` of procedure gds.graph.drop() is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL gds.graph.drop($name, false)'


Received notification from DBMS server: <GqlStatusObject gql_status='01N03', status_description='warn: procedure field deprecated. The field `schema` of procedure gds.graph.drop() is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL gds.graph.drop($name, false)'


Received notification from DBMS server: <GqlStatusObject gql_status='01N03', status_description='warn: procedure field deprecated. The field `schema` of procedure gds.graph.drop() is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL gds.graph.drop($name, false)'


2026-07-06 14:23:03.228 | INFO     | knowledge_graph_foundry.extraction.embeddings:_embed_bedrock:91 - Embeddings: 1/1
2026-07-06 14:23:03.230 | INFO     | knowledge_graph_foundry.extraction.embeddings:generate_embeddings:199 - Generated embeddings for 1/1 entities via bedrock (0 cache hits)
2026-07-06 14:23:03.265 | WARNING  | knowledge_graph_foundry.graph.graphrag:ppr_query:193 - PPR query fell back (graph issue): {neo4j_code: Neo.ClientError.Procedure.ProcedureCallFailed} {message: Failed to invoke procedure `gds.graph.project`: Caused by: java.lang.IllegalArgumentException: Invalid node projection, one or more labels not found: 'Entity, Chunk'} {gql_status: 50N42} {gql_status_description: error: general processing exception - unexpected error. Unexpected error has occurred. See debug log for details.}
2026-07-06 14:23:03.282 | INFO     | knowledge_graph_foundry.extraction.embeddings:generate_embeddings:188 - Embeddings: 1/1 from cache
2026-07-06 14:23:03.294 | WARNING  | knowledge_

...

Received notification from DBMS server: <GqlStatusObject gql_status='01N03', status_description='warn: procedure field deprecated. The field `schema` of procedure gds.graph.drop() is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL gds.graph.drop($name, false)'


Received notification from DBMS server: <GqlStatusObject gql_status='01N03', status_description='warn: procedure field deprecated. The field `schema` of procedure gds.graph.drop() is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL gds.graph.drop($name, false)'


Received notification from DBMS server: <GqlStatusObject gql_status='01N03', status_description='warn: procedure field deprecated. The field `schema` of procedure gds.graph.drop() is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL gds.graph.drop($name, false)'


P23 1.0 False
2026-07-06 14:25:53.850 | INFO     | knowledge_graph_foundry.extraction.embeddings:generate_embeddings:188 - Embeddings: 1/1 from cache
2026-07-06 14:25:53.863 | WARNING  | knowledge_graph_foundry.graph.graphrag:ppr_query:193 - PPR query fell back (graph issue): {neo4j_code: Neo.ClientError.Procedure.ProcedureCallFailed} {message: Failed to invoke procedure `gds.graph.project`: Caused by: java.lang.IllegalArgumentException: Invalid node projection, one or more labels not found: 'Entity, Chunk'} {gql_status: 50N42} {gql_status_description: error: general processing exception - unexpected error. Unexpected error has occurred. See debug log for details.}
2026-07-06 14:25:53.881 | INFO     | knowledge_graph_foundry.extraction.embeddings:generate_embeddings:188 - Embeddings: 1/1 from cache
2026-07-06 14:25:53.893 | WARNING  | knowledge_graph_foundry.graph.graphrag:ppr_query:193 - PPR query fell back (graph issue): {neo4j_code: Neo.ClientError.Procedure.ProcedureCallFailed} {me

P24 0.0 True
H14: 0.4375 -> 0.4375 | acc 0.75 -> 0.75


In [4]:
# H15: comparison decomposition off vs on (comparison probes only)
s = base_settings(); s.graphrag.decompose_comparisons = False
h15_off = run(s, comparisons, 'H15_decompose_off')
s = base_settings(); s.graphrag.decompose_comparisons = True
h15_on = run(s, comparisons, 'H15_decompose_on')
print('H15 comparisons:', h15_off['accuracy'], '->', h15_on['accuracy'],
      '| recall', h15_off['recall'], '->', h15_on['recall'])

Received notification from DBMS server: <GqlStatusObject gql_status='01N03', status_description='warn: procedure field deprecated. The field `schema` of procedure gds.graph.drop() is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL gds.graph.drop($name, false)'


Received notification from DBMS server: <GqlStatusObject gql_status='01N03', status_description='warn: procedure field deprecated. The field `schema` of procedure gds.graph.drop() is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL gds.graph.drop($name, false)'


Received notification from DBMS server: <GqlStatusObject gql_status='01N03', status_description='warn: procedure field deprecated. The field `schema` of procedure gds.graph.drop() is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL gds.graph.drop($name, false)'


Received notification from DBMS server: <GqlStatusObject gql_status='01N03', status_description='warn: procedure field deprecated. The field `schema` of procedure gds.graph.drop() is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL gds.graph.drop($name, false)'


2026-07-06 14:25:58.221 | INFO     | knowledge_graph_foundry.extraction.embeddings:generate_embeddings:188 - Embeddings: 1/1 from cache
2026-07-06 14:25:58.238 | WARNING  | knowledge_graph_foundry.graph.graphrag:ppr_query:193 - PPR query fell back (graph issue): {neo4j_code: Neo.ClientError.Procedure.ProcedureCallFailed} {message: Failed to invoke procedure `gds.graph.project`: Caused by: java.lang.IllegalArgumentException: Invalid node projection, one or more labels not found: 'Entity, Chunk'} {gql_status: 50N42} {gql_status_description: error: general processing exception - unexpected error. Unexpected error has occurred. See debug log for details.}
2026-07-06 14:25:58.258 | INFO     | knowledge_graph_foundry.extraction.embeddings:generate_embeddings:188 - Embeddings: 1/1 from cache
2026-07-06 14:25:58.269 | WARNING  | knowledge_graph_foundry.graph.graphrag:ppr_query:193 - PPR query fell back (graph issue): {neo4j_code: Neo.ClientError.Procedure.ProcedureCallFailed} {message: Failed 

...

Received notification from DBMS server: <GqlStatusObject gql_status='01N03', status_description='warn: procedure field deprecated. The field `schema` of procedure gds.graph.drop() is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL gds.graph.drop($name, false)'


Received notification from DBMS server: <GqlStatusObject gql_status='01N03', status_description='warn: procedure field deprecated. The field `schema` of procedure gds.graph.drop() is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL gds.graph.drop($name, false)'


Received notification from DBMS server: <GqlStatusObject gql_status='01N03', status_description='warn: procedure field deprecated. The field `schema` of procedure gds.graph.drop() is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL gds.graph.drop($name, false)'


P17 0.5 True
2026-07-06 14:26:52.575 | INFO     | knowledge_graph_foundry.extraction.embeddings:generate_embeddings:188 - Embeddings: 1/1 from cache
2026-07-06 14:26:52.593 | WARNING  | knowledge_graph_foundry.graph.graphrag:ppr_query:193 - PPR query fell back (graph issue): {neo4j_code: Neo.ClientError.Procedure.ProcedureCallFailed} {message: Failed to invoke procedure `gds.graph.project`: Caused by: java.lang.IllegalArgumentException: Invalid node projection, one or more labels not found: 'Entity, Chunk'} {gql_status: 50N42} {gql_status_description: error: general processing exception - unexpected error. Unexpected error has occurred. See debug log for details.}
2026-07-06 14:26:52.614 | INFO     | knowledge_graph_foundry.extraction.embeddings:generate_embeddings:188 - Embeddings: 1/1 from cache
2026-07-06 14:26:52.627 | WARNING  | knowledge_graph_foundry.graph.graphrag:ppr_query:193 - PPR query fell back (graph issue): {neo4j_code: Neo.ClientError.Procedure.ProcedureCallFailed} {mes

P18 0.0 True
H15 comparisons: 0.875 -> 0.875 | recall 0.1875 -> 0.1875


In [5]:
# H16: head+tail interleave off vs on (all answerable, accuracy focus)
s = base_settings(); s.graphrag.context_head_tail = False
h16_off = run(s, answerable, 'H16_headtail_off', with_recall=False)
s = base_settings(); s.graphrag.context_head_tail = True
h16_on = run(s, answerable, 'H16_headtail_on', with_recall=False)
print('H16:', h16_off['accuracy'], '->', h16_on['accuracy'])

Received notification from DBMS server: <GqlStatusObject gql_status='01N03', status_description='warn: procedure field deprecated. The field `schema` of procedure gds.graph.drop() is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL gds.graph.drop($name, false)'


Received notification from DBMS server: <GqlStatusObject gql_status='01N03', status_description='warn: procedure field deprecated. The field `schema` of procedure gds.graph.drop() is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL gds.graph.drop($name, false)'


2026-07-06 14:26:56.142 | INFO     | knowledge_graph_foundry.extraction.embeddings:generate_embeddings:188 - Embeddings: 1/1 from cache
2026-07-06 14:26:56.161 | WARNING  | knowledge_graph_foundry.graph.graphrag:ppr_query:193 - PPR query fell back (graph issue): {neo4j_code: Neo.ClientError.Procedure.ProcedureCallFailed} {message: Failed to invoke procedure `gds.graph.project`: Caused by: java.lang.IllegalArgumentException: Invalid node projection, one or more labels not found: 'Entity, Chunk'} {gql_status: 50N42} {gql_status_description: error: general processing exception - unexpected error. Unexpected error has occurred. See debug log for details.}


Received notification from DBMS server: <GqlStatusObject gql_status='01N03', status_description='warn: procedure field deprecated. The field `schema` of procedure gds.graph.drop() is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL gds.graph.drop($name, false)'


Received notification from DBMS server: <GqlStatusObject gql_status='01N03', status_description='warn: procedure field deprecated. The field `schema` of procedure gds.graph.drop() is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL gds.graph.drop($name, false)'


...

P22 None False
2026-07-06 14:29:33.514 | INFO     | knowledge_graph_foundry.extraction.embeddings:generate_embeddings:188 - Embeddings: 1/1 from cache
2026-07-06 14:29:33.527 | WARNING  | knowledge_graph_foundry.graph.graphrag:ppr_query:193 - PPR query fell back (graph issue): {neo4j_code: Neo.ClientError.Procedure.ProcedureCallFailed} {message: Failed to invoke procedure `gds.graph.project`: Caused by: java.lang.IllegalArgumentException: Invalid node projection, one or more labels not found: 'Entity, Chunk'} {gql_status: 50N42} {gql_status_description: error: general processing exception - unexpected error. Unexpected error has occurred. See debug log for details.}


Received notification from DBMS server: <GqlStatusObject gql_status='01N03', status_description='warn: procedure field deprecated. The field `schema` of procedure gds.graph.drop() is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL gds.graph.drop($name, false)'


Received notification from DBMS server: <GqlStatusObject gql_status='01N03', status_description='warn: procedure field deprecated. The field `schema` of procedure gds.graph.drop() is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL gds.graph.drop($name, false)'


P23 None False
2026-07-06 14:29:36.831 | INFO     | knowledge_graph_foundry.extraction.embeddings:generate_embeddings:188 - Embeddings: 1/1 from cache
2026-07-06 14:29:36.847 | WARNING  | knowledge_graph_foundry.graph.graphrag:ppr_query:193 - PPR query fell back (graph issue): {neo4j_code: Neo.ClientError.Procedure.ProcedureCallFailed} {message: Failed to invoke procedure `gds.graph.project`: Caused by: java.lang.IllegalArgumentException: Invalid node projection, one or more labels not found: 'Entity, Chunk'} {gql_status: 50N42} {gql_status_description: error: general processing exception - unexpected error. Unexpected error has occurred. See debug log for details.}


P24 None True
H16: 0.75 -> 0.75


In [6]:
# persist
results = {r['label']: r for r in
           (h14_off, h14_on, h15_off, h15_on, h16_off, h16_on)}
for r in results.values():
    print(f"{r['label']:<20} recall={r['recall']} acc={r['accuracy']:.3f}")
stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d-%H%M%S')
out = Path('../reports') / f'probe-eval-r03-{stamp}.json'
out.write_text(json.dumps(results, indent=2, default=str))
print('saved', out)

H14_seeding_off      recall=0.4375 acc=0.750
H14_seeding_on       recall=0.4375 acc=0.750
H15_decompose_off    recall=0.1875 acc=0.875
H15_decompose_on     recall=0.1875 acc=0.875
H16_headtail_off     recall=None acc=0.750
H16_headtail_on      recall=None acc=0.750
saved ../reports/probe-eval-r03-20260706-122941.json
